```{index} privacy; security
```

(privacy-security)=

# Privacy and Security in Code

```{contents}
:local:
:depth: 2
```

Business programs handle **personal data**: names, email addresses, phone numbers, card numbers, birth dates, and purchase histories. Personal data is any information that identifies a person or can be linked to one. Mishandling it harms the people it describes and exposes the organization to legal penalties. This section turns privacy and security principles into small coding habits you can use now.

```{index} privacy; law
```

## Personal Data and the Law

Privacy laws differ by place, but they share common ideas. Three examples:

- **GDPR** (EU Regulation 2016/679, applies from May 25, 2018) sets principles for processing personal data, including *purpose limitation* (use data only for the stated purpose), *data minimisation* (collect only what is needed), and *storage limitation* (do not keep it longer than needed).[^gdpr]
- **CCPA** (California Consumer Privacy Act, in effect since January 1, 2020, and expanded by the CPRA from January 1, 2023) gives California residents rights to know about, delete, and opt out of the sale of their personal information.[^ccpa]
- **FERPA** (Family Educational Rights and Privacy Act of 1974) protects the privacy of student education records in the United States, including your grades in this course.[^ferpa]

You do not need to be a lawyer to follow the habits these laws have in common: **collect less, show less, and keep less.**

```{index} masking; personal data
```

## Show Less: Masking

Many screens and documents need only enough of a value to recognize it. A support agent confirming an account needs to see `a***@example.com`, not the full address. Masking uses the string methods from {ref}`string-methods`:

In [ ]:
using System;

// Show enough to recognize the address, hide the rest.
static string MaskEmail(string email)
{
    int at = email.IndexOf('@');
    if (at < 1)
    {
        return "(invalid email)";
    }
    return email.Substring(0, 1) + "***" + email.Substring(at);
}

Console.WriteLine(MaskEmail("ana.diaz@example.com"));   // a***@example.com
Console.WriteLine(MaskEmail("@example.com"));           // (invalid email)

Receipts and order confirmations conventionally show only the last four digits of a card:

In [ ]:
using System;

// Receipts and support screens need only the last four digits of a card.
static string MaskCard(string cardNumber)
{
    string digits = cardNumber.Replace(" ", "").Replace("-", "");
    if (digits.Length < 4)
    {
        return "(invalid card number)";
    }
    return "**** **** **** " + digits.Substring(digits.Length - 4);
}

Console.WriteLine(MaskCard("4111 1111 1111 1234"));   // **** **** **** 1234
Console.WriteLine(MaskCard("12"));                    // (invalid card number)

Masking is for display. A program that does not need a value at all should not collect or store it in the first place.

```{index} untrusted input; validation
```

## Treat Every Input as Untrusted

Every value that comes from outside the program, such as keyboard input, a file, a web form, or another system, may be wrong by accident or on purpose. Check both the **type** and the **business rule** before using it. `int.TryParse` from Chapter 6 handles the type; an `if` handles the range:

In [ ]:
using System;

// Treat every input as untrusted: check the type and the business range.
static string CheckQuantity(string input)
{
    if (!int.TryParse(input, out int quantity))
    {
        return $"Rejected \"{input}\": not a whole number.";
    }
    if (quantity < 1 || quantity > 500)
    {
        return $"Rejected {quantity}: quantity must be from 1 to 500.";
    }
    return $"Accepted {quantity}.";
}

Console.WriteLine(CheckQuantity("12"));
Console.WriteLine(CheckQuantity("12x"));
Console.WriteLine(CheckQuantity("-3"));
Console.WriteLine(CheckQuantity("1000000"));

Notice that the messages say *what* was wrong without crashing the program. A missing check can do more than produce a wrong answer: the OWASP Top 10, a widely used list of the most serious web application security risks, includes Broken Access Control (A01) and Injection (A05) in its 2025 edition.[^owasp]

```{index} injection; SQL injection
```

## Injection

**Injection** happens when a program builds a command, such as a database query, by gluing user input into the command text. Input crafted by an attacker then becomes part of the command:

In [ ]:
using System;

// Building a database query by gluing user input into the text is dangerous.
static string UnsafeQuery(string customerName)
{
    return "SELECT * FROM Customers WHERE Name = '" + customerName + "'";
}

Console.WriteLine(UnsafeQuery("Ana Diaz"));
Console.WriteLine(UnsafeQuery("x' OR '1'='1"));   // now matches every customer

The second query no longer looks for one customer: `OR '1'='1'` is always true, so it matches every row and could expose every customer's data. The fix is never to build commands from input text. Chapter 11 shows *parameterized commands*, which send the input separately from the command so it is always treated as data.

```{index} logging; secrets
```

## Logging Without Leaking

Logs record what a program did so people can investigate problems later. Logs are also read by many people and kept for a long time, so they must not contain secrets such as passwords or full card numbers:

In [ ]:
using System;

// Log what happened and who did it; never log secrets.
static string LogLine(string userId, string action, bool succeeded)
{
    string result = succeeded ? "succeeded" : "failed";
    return $"{DateTime.UtcNow:yyyy-MM-dd HH:mm} UTC user={userId} action={action} result={result}";
}

string userId = "u1042";
string password = "correct-horse-battery";   // typed by the user; never written to a log

bool signedIn = password.Length >= 12;   // stand-in for a real password check
Console.WriteLine(LogLine(userId, "sign-in", signedIn));

Three related principles guide secure design:

- **Confidentiality, integrity, availability.** Keep data from people who should not see it, keep it from being changed improperly, and keep the system usable for those who need it.
- **Least privilege.** Give each user and program only the access its task requires. A report that only reads orders should not be able to delete them.
- **Never store passwords.** Systems store a salted, slow *hash* of a password: a one-way value that can check a password without revealing it. Use a well-tested library for this; never write your own.

```{index} privacy; summary
```

## Summary

Personal data deserves care in every line of code that touches it. Collect, show, and keep only what a task needs; mask values for display; check every input for type and business rules; never build commands from input text; and keep secrets out of logs. These habits match the principles that privacy laws such as GDPR, CCPA, and FERPA share.

```{rubric} Footnotes
```

[^gdpr]: European Parliament and Council (2016). *Regulation (EU) 2016/679 (General Data Protection Regulation)*, Article 5. <https://eur-lex.europa.eu/eli/reg/2016/679/oj>

[^ccpa]: State of California, Office of the Attorney General. *California Consumer Privacy Act (CCPA)*. <https://oag.ca.gov/privacy/ccpa>

[^ferpa]: U.S. Department of Education. *Family Educational Rights and Privacy Act (FERPA)*. <https://studentprivacy.ed.gov/ferpa>

[^owasp]: OWASP Foundation (2025). *OWASP Top 10:2025*. <https://top10.owasp.org/2025/>. In the 2021 edition, Injection was ranked A03.